# 04 · Evaluate, benchmark and export

Three ways to evaluate a checkpoint, and how to release it:

1. **Validation loss / perplexity** on held-out shards (cheap; the number for scaling laws)
2. **lm-evaluation-harness** (HellaSwag, ARC, PIQA, Winogrande, MMLU, GSM8K, ...) directly on the JAX model
3. **Export to Hugging Face format** and verify that `transformers` reproduces the JAX logits — that is the artifact we publish.

In [ ]:
# --- GW1B setup cell (same in every notebook) -------------------------------------------------
import os, sys, json, time, pathlib
GW1B_HOME    = os.environ.get("GW1B_HOME", "/SEAS/groups/gw1b/gw1b-f2026")
GW1B_SCRATCH = os.environ.get("GW1B_SCRATCH", "/lustre/groups/gw1b")
GW1B_GROUP   = os.environ.get("GW1B_GROUP", "/SEAS/groups/gw1b")
USER_DIR     = os.path.join(GW1B_SCRATCH, "users", os.environ.get("USER", "student"))
os.makedirs(USER_DIR, exist_ok=True)
if GW1B_HOME not in sys.path:
    sys.path.insert(0, GW1B_HOME)        # the shared gw1b package
import jax, jax.numpy as jnp, numpy as np
print("JAX", jax.__version__, "| backend:", jax.default_backend(), "| devices:", jax.devices())
print("your scratch dir:", USER_DIR)


In [ ]:
from gw1b.evaluate import load_run, eval_loss
run_dir = os.path.join(USER_DIR, "runs", "nb03-tiny")        # from notebook 03 (or any run directory)
model, cfg, step = load_run(run_dir, shard_params=False)
res = eval_loss(model, cfg.data.val_dir, cfg.data.seq_len, batch_size=8, n_batches=10)
print(f"step {step}: val loss {res['loss']:.3f}, perplexity {res['ppl']:.1f} on {res['tokens_evaluated']:,} tokens")

## 2. lm-evaluation-harness
Works on any checkpoint, no conversion. Benchmarks are downloaded from the Hugging Face Hub the first time
(do that once on a login node so the compute nodes find them in `$HF_HOME`).

In [ ]:
from gw1b.lm_eval_adapter import GW1BLM
from gw1b.tokenizer import Tokenizer
lm = GW1BLM(model, Tokenizer(cfg.data.tokenizer), batch_size=8)
try:
    import lm_eval
    out = lm_eval.simple_evaluate(model=lm, tasks=["hellaswag"], limit=50, log_samples=False)
    results = out["results"] if isinstance(out, dict) else out.results
    print({k: v for k, v in results["hellaswag"].items() if isinstance(v, float)})
except Exception as e:
    print("lm-eval could not run here (offline or task cache missing):", str(e)[:200])
    print("batch version:  gw1b run --gpus 1 -m gw1b.lm_eval_adapter --run", run_dir, "--tasks hellaswag,arc_easy,piqa --limit 500")

## 3. Export to Hugging Face format and verify

In [ ]:
from gw1b.export_hf import export, verify
out_dir = os.path.join(USER_DIR, "release", "nb03-tiny-hf")
export(model, out_dir, tokenizer_path=cfg.data.tokenizer, dtype="float32")
diff = verify(model, out_dir)          # loads with transformers on CPU and compares log-probs
print(sorted(os.listdir(out_dir)))

## 4. The numbers every experiment reports

In [ ]:
import pandas as pd
s = json.load(open(os.path.join(run_dir, "summary.json")))
pd.DataFrame([{ "run": s["run"], "params (M)": s["n_params"]/1e6, "tokens (M)": s["tokens"]/1e6, "FLOPs": f"{s['flops']:.2e}",
                "GPU-hours": round(s["gpu_hours"], 3), "USD": round(s["usd_equivalent"], 2),
                "val loss": round(res["loss"], 3), "val ppl": round(res["ppl"], 1)}]).T

## 5. Releasing GW-1B (end of semester)
```
python -m gw1b.export_hf --run $GW1B_SCRATCH/checkpoints/gw1b-base --out $GW1B_GROUP/release/GW-1B-Base --verify
huggingface-cli upload gwu-gw1b/GW-1B-Base $GW1B_GROUP/release/GW-1B-Base      # from a login node
```
plus the tokenizer, the data manifest (`manifest.json`), configs, `metrics.jsonl` and the model card.